# Practical 9: YOLO (YOLOv8-cls) Classification on the Fruits & Vegetables Dataset

**Aim:** To classify images of 36 fruit and vegetable types using a YOLOv8 classification model, and to handle practical issues such as read-only dataset folders (Kaggle) and slow input/output.

> Reconstructed code (the source PDF has no code). Not executed by the author - run cells in order (Kernel -> Restart & Run All) and check dataset paths in the first cells. Expected values: Top-1 accuracy ~92-96%, Top-5 ~98-99%.; 36 class folders, NO bounding boxes -> classification only.

## Cell 1 - Install and import

**What it does:**
- Installs and imports Ultralytics and helper libraries.

*Expected output (illustrative):* No output.

In [ ]:
!pip install -q ultralytics

import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
from ultralytics import YOLO

IMG_EXT = {".jpg", ".jpeg", ".png"}

## Cell 2 - Copy the dataset to a writable folder

**What it does:**
- On Kaggle use `SRC = Path('/kaggle/input/fruit-and-vegetable-image-recognition')` and `DST = Path('/kaggle/working/fruits_veg')`.
- `shutil.copytree` copies the whole dataset to a place where Ultralytics may write its cache files. This prevents the `[Errno 30] Read-only file system` error (from your PDF). The same fix applies to OneDrive/read-only folders: use e.g. `C:\datasets\`.

*Expected output (illustrative):* Folders: ['test', 'train', 'validation']

In [ ]:
SRC = Path("fruit-and-vegetable-image-recognition")
DST = Path("fruits_veg_data")

if not DST.exists():
    shutil.copytree(SRC, DST)

print("Folders:", sorted(d.name for d in DST.iterdir() if d.is_dir()))

## Cell 3 - Inspect classes and image counts

**What it does:**
- Confirms there are 36 class folders and counts images. `VAL_DIR` is `validation` (Kaggle name) or `val`; Ultralytics accepts both names.

*Expected output (illustrative):* Number of classes: 36
Roughly 3,100 train and 350 validation images (counts depend on the dataset version; total ~3.8k per your PDF) followed by the list of 36 names (apple, banana, beetroot, bell pepper, cabbage, capsicum, ...).

In [ ]:
train_dir = DST / "train"
VAL_DIR = DST / "validation" if (DST / "validation").exists() else DST / "val"

class_names = sorted(d.name for d in train_dir.iterdir() if d.is_dir())
print("Number of classes:", len(class_names))

def count_images(folder):
    return sum(1 for f in folder.rglob("*") if f.suffix.lower() in IMG_EXT)

print("Train images     :", count_images(train_dir))
print("Validation images:", count_images(VAL_DIR))
print(class_names)

## Cell 4 - Train YOLOv8n-cls

**What it does:**
- `cache='ram'` loads all (resized) images into memory once - the following epochs are not slowed by disk reads.
- `batch=128` is larger than usual (your PDF recommends 128+ for YOLO/transfer learning). If you get CUDA out-of-memory, reduce it to 64 or 32.
- `epochs=15` and `workers=2` are reconstructed values; increase epochs for slightly higher accuracy.

*Expected output (illustrative):* Per-epoch table (loss, top1/top5 accuracy). Final top-1 about 0.92-0.96.

In [ ]:
model = YOLO("yolov8n-cls.pt")

results = model.train(
    data=str(DST.resolve()),
    epochs=15,
    imgsz=224,
    batch=128,
    cache="ram",
    workers=2,
)

save_dir = Path(model.trainer.save_dir)
print("Results saved in:", save_dir)

## Cell 5 - Plot the training curves

**What it does:**
- Loss falls and accuracies rise; Top-5 stays above Top-1.

*Expected output (illustrative):* Falling loss; Top-1 ~0.92-0.96, Top-5 ~0.98-0.99 at the end.

In [ ]:
df = pd.read_csv(save_dir / "results.csv")
df.columns = df.columns.str.strip()

plt.figure(figsize=(11, 4))

plt.subplot(1, 2, 1)
plt.plot(df["epoch"], df["train/loss"], label="Train loss")
plt.plot(df["epoch"], df["val/loss"], label="Validation loss")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(df["epoch"], df["metrics/accuracy_top1"], label="Top-1 accuracy")
plt.plot(df["epoch"], df["metrics/accuracy_top5"], label="Top-5 accuracy")
plt.title("Validation accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.show()

## Cell 6 - Validate: Top-1 and Top-5 accuracy

**What it does:**
- Evaluates the best checkpoint on the validation folder.

*Expected output (illustrative):* Top-1 ~0.92-0.96, Top-5 ~0.98-0.99 (from your PDF).

In [ ]:
best = YOLO(str(save_dir / "weights" / "best.pt"))

metrics = best.val(data=str(DST.resolve()), imgsz=224)
print("Top-1 accuracy:", round(metrics.top1, 4))
print("Top-5 accuracy:", round(metrics.top5, 4))

## Cell 7 - Confusion matrix and precision / recall / F1

**What it does:**
- Predicts all validation images (in batches of 64 to save memory) and builds a 36 x 36 confusion matrix (numbers hidden for readability).
- The report lists precision/recall/F1 for each of the 36 classes; low scores point at the pepper family and leafy greens.

*Expected output (illustrative):* A large 36 x 36 matrix with a bright diagonal, and a 36-row classification report (accuracy ~0.92-0.96).

In [ ]:
CM_SIZE = 14
INCLUDE_VALUES = False

names = best.names
name_to_id = {v: k for k, v in names.items()}

val_files, y_true = [], []
for class_dir in sorted(VAL_DIR.iterdir()):
    if class_dir.is_dir():
        for f in sorted(class_dir.iterdir()):
            if f.suffix.lower() in IMG_EXT:
                val_files.append(str(f))
                y_true.append(name_to_id[class_dir.name])

y_pred = []
for start in range(0, len(val_files), 64):
    batch_results = best.predict(val_files[start:start + 64], imgsz=224, verbose=False)
    y_pred.extend(int(r.probs.top1) for r in batch_results)

labels = list(range(len(names)))
label_names = [names[i] for i in labels]

cm = confusion_matrix(y_true, y_pred, labels=labels)
fig, ax = plt.subplots(figsize=(CM_SIZE, CM_SIZE))
ConfusionMatrixDisplay(cm, display_labels=label_names).plot(
    ax=ax, cmap="Blues", xticks_rotation=90, include_values=INCLUDE_VALUES, colorbar=False)
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, labels=labels, target_names=label_names, digits=3))

## Cell 8 - Predict a single image (Top-5 probabilities)

**What it does:**
- Prints the five most likely classes with their probabilities (this is how Top-5 accuracy is defined).

*Expected output (illustrative):* Five lines such as 'apple 0.97, pomegranate 0.01, ...' and the picture.

In [ ]:
sample = val_files[0]
r = best.predict(sample, imgsz=224, verbose=False)[0]

print("True class:", Path(sample).parent.name)
for idx, conf in zip(r.probs.top5, r.probs.top5conf.tolist()):
    print(" ", r.names[idx], round(conf, 3))

plt.imshow(Image.open(sample))
plt.title("Predicted: " + r.names[r.probs.top1])
plt.axis("off")
plt.show()

## Result / Conclusion

YOLOv8n-cls fine-tuned from ImageNet weights classified 36 fruit and vegetable classes with ~92-96% Top-1 accuracy; copying the data to a writable folder and using RAM caching with batch 128 avoided I/O problems.